In [ ]:
from pathlib import Path
import json
import random
import numpy as np

# PROJECT PATHS


# segmentation_model.ipynb is inside:
# IBY_PROJECT/pipelines/

PROJECT_ROOT = Path("..")

DATASET_A_DIR = PROJECT_ROOT / "notebooks" / "dataset_A" / "dataset_a_combined"
DATASET_B_DIR = PROJECT_ROOT / "notebooks" / "dataset_B"

ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
MODELS_DIR = ARTIFACTS_DIR / "models"
PREPROCESSING_DIR = ARTIFACTS_DIR / "preprocessing"
METADATA_DIR = ARTIFACTS_DIR / "metadata"



# CREATE ARTIFACT DIRECTORIES


for directory in [
    ARTIFACTS_DIR,
    MODELS_DIR,
    PREPROCESSING_DIR,
    METADATA_DIR
]:
    directory.mkdir(parents=True, exist_ok=True)



# REPRODUCIBILITY


RANDOM_SEED = 42

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)



# CONFIGURATION


CONFIG = {
    "project_root": str(PROJECT_ROOT.resolve()),
    "dataset_a_dir": str(DATASET_A_DIR.resolve()),
    "dataset_b_dir": str(DATASET_B_DIR.resolve()),
    "artifacts_dir": str(ARTIFACTS_DIR.resolve()),
    "models_dir": str(MODELS_DIR.resolve()),
    "preprocessing_dir": str(PREPROCESSING_DIR.resolve()),
    "metadata_dir": str(METADATA_DIR.resolve()),
    "random_seed": RANDOM_SEED,
}



# DISPLAY CONFIGURATION


print("=" * 70)
print("SEGMENTATION DEVELOPMENT — CONFIGURATION")
print("=" * 70)

for key, value in CONFIG.items():
    print(f"{key}: {value}")

print("\nPath checks:")
print(f"Dataset A exists: {DATASET_A_DIR.exists()}")
print(f"Dataset B exists: {DATASET_B_DIR.exists()}")
print(f"Artifacts directory: {ARTIFACTS_DIR.resolve()}")

SEGMENTATION DEVELOPMENT — CONFIGURATION
project_root: C:\Users\Shambhavi Singh\OneDrive\Desktop\IBY_PROJECT
dataset_a_dir: C:\Users\Shambhavi Singh\OneDrive\Desktop\IBY_PROJECT\notebooks\dataset_A\dataset_a_combined
dataset_b_dir: C:\Users\Shambhavi Singh\OneDrive\Desktop\IBY_PROJECT\notebooks\dataset_B
artifacts_dir: C:\Users\Shambhavi Singh\OneDrive\Desktop\IBY_PROJECT\artifacts
models_dir: C:\Users\Shambhavi Singh\OneDrive\Desktop\IBY_PROJECT\artifacts\models
preprocessing_dir: C:\Users\Shambhavi Singh\OneDrive\Desktop\IBY_PROJECT\artifacts\preprocessing
metadata_dir: C:\Users\Shambhavi Singh\OneDrive\Desktop\IBY_PROJECT\artifacts\metadata
random_seed: 42

Path checks:
Dataset A exists: True
Dataset B exists: True
Artifacts directory: C:\Users\Shambhavi Singh\OneDrive\Desktop\IBY_PROJECT\artifacts


In [2]:
# Load and prepare all Dataset A raw events

import json
import pandas as pd


def load_jsonl_events(events_file):
    """
    Load one events.jsonl file into a list of dictionaries.
    Invalid/empty lines are skipped safely.
    """
    records = []

    with open(events_file, "r", encoding="utf-8") as f:
        for line_number, line in enumerate(f, start=1):
            line = line.strip()

            if not line:
                continue

            try:
                records.append(json.loads(line))
            except json.JSONDecodeError:
                print(
                    f"Warning: could not parse line {line_number} "
                    f"in {events_file}"
                )

    return records


# Find every events.jsonl file in Dataset A


event_files_a = sorted(DATASET_A_DIR.rglob("events.jsonl"))

print("=" * 70)
print("DATASET A — RAW EVENT LOADING")
print("=" * 70)

print(f"Event files found: {len(event_files_a)}")

#Load all events

all_event_records = []

for event_file in event_files_a:
    records = load_jsonl_events(event_file)

    all_event_records.extend(records)


print(f"Raw events loaded: {len(all_event_records):,}")



# Convert to DataFrame


ALL_EVENTS_A = pd.json_normalize(all_event_records)


# Sort chronologically


if "timestamp_ms" not in ALL_EVENTS_A.columns:
    raise ValueError(
        "timestamp_ms column was not found in the raw event data."
    )

ALL_EVENTS_A = (
    ALL_EVENTS_A
    .sort_values(
        by=["session_id", "timestamp_ms"],
        kind="stable"
    )
    .reset_index(drop=True)
)



# Basic information


print("\nDataFrame shape:")
print(ALL_EVENTS_A.shape)

print("\nSessions:")
print(ALL_EVENTS_A["session_id"].nunique())

print("\nEvent types:")
print(ALL_EVENTS_A["event_type"].value_counts().head(15))

print("\nFirst 5 chronological events:")
print(
    ALL_EVENTS_A[
        [
            "session_id",
            "timestamp_ms",
            "timestamp_iso",
            "event_type"
        ]
    ].head().to_string(index=False)
)

DATASET A — RAW EVENT LOADING
Event files found: 117
Raw events loaded: 162,768

DataFrame shape:
(162768, 410)

Sessions:
63

Event types:
event_type
app_switch                50588
keystroke                 38717
screenshot_smart          34580
shortcut                  13668
mouse_click                6177
browser_click              5365
clipboard_change           5198
mouse_scroll               3180
browser_form_input         1805
browser_navigation         1725
window_title_change         451
window_state_change         232
browser_error               200
extension_disconnected      172
browser_alert               137
Name: count, dtype: int64

First 5 chronological events:
                         session_id  timestamp_ms            timestamp_iso       event_type
ses_20260630-121953-LAPTOP-R36BQBTE 1782821993821 2026-06-30T12:19:53.821Z    session_start
ses_20260630-121953-LAPTOP-R36BQBTE 1782821994435 2026-06-30T12:19:54.435Z      mouse_click
ses_20260630-121953-LAPTOP-R36BQBTE 

In [21]:
# Build event-level features

print("=" * 70)
print("SEGMENTATION MODEL — EVENT FEATURE ENGINEERING")
print("=" * 70)

# Start from raw events loaded in Cell 2
features = ALL_EVENTS_A.copy()

# Timestamp
features["timestamp"] = pd.to_datetime(
    features["timestamp_iso"],
    utc=True,
    errors="coerce"
)

# Sort chronologically within each session
features = features.sort_values(
    ["session_id", "timestamp"],
    kind="stable"
).reset_index(drop=True)



features["time_since_previous_event"] = (
    features
    .groupby("session_id")["timestamp"]
    .diff()
    .dt.total_seconds()
)

session_start = (
    features
    .groupby("session_id")["timestamp"]
    .transform("min")
)

features["time_since_session_start"] = (
    features["timestamp"] - session_start
).dt.total_seconds()



features["is_click"] = (
    features["event_type"]
    .astype(str)
    .str.contains("click", case=False, na=False)
    .astype(int)
)

features["is_keystroke"] = (
    features["event_type"]
    .astype(str)
    .eq("keystroke")
    .astype(int)
)

features["is_app_switch"] = (
    features["event_type"]
    .astype(str)
    .eq("app_switch")
    .astype(int)
)

features["is_screenshot"] = (
    features["event_type"]
    .astype(str)
    .str.contains("screenshot", case=False, na=False)
    .astype(int)
)

features["is_clipboard"] = (
    features["event_type"]
    .astype(str)
    .str.contains("clipboard", case=False, na=False)
    .astype(int)
)

features["is_browser"] = (
    features["event_type"]
    .astype(str)
    .str.contains("browser", case=False, na=False)
    .astype(int)
)

# Clean temporal gaps
features["time_since_previous_event"] = (
    features["time_since_previous_event"]
    .fillna(0)
    .clip(lower=0)
)

# Check


print("\nFEATURE TABLE")
print("-------------")

print("Rows:", len(features))
print("Columns:", len(features.columns))
print("Sessions:", features["session_id"].nunique())

print("\nSelected features:")
print(
    features[
        [
            "session_id",
            "timestamp",
            "event_type",
            "time_since_previous_event",
            "time_since_session_start",
            "is_click",
            "is_keystroke",
            "is_app_switch",
            "is_screenshot",
            "is_clipboard",
            "is_browser"
        ]
    ].head(10).to_string(index=False)
)

print("\nFEATURE ENGINEERING COMPLETE")

SEGMENTATION MODEL — EVENT FEATURE ENGINEERING

FEATURE TABLE
-------------
Rows: 162768
Columns: 419
Sessions: 63

Selected features:
                         session_id                        timestamp       event_type  time_since_previous_event  time_since_session_start  is_click  is_keystroke  is_app_switch  is_screenshot  is_clipboard  is_browser
ses_20260630-121953-LAPTOP-R36BQBTE 2026-06-30 12:19:53.821000+00:00    session_start                      0.000                     0.000         0             0              0              0             0           0
ses_20260630-121953-LAPTOP-R36BQBTE 2026-06-30 12:19:54.435000+00:00      mouse_click                      0.614                     0.614         1             0              0              0             0           0
ses_20260630-121953-LAPTOP-R36BQBTE 2026-06-30 12:19:54.465000+00:00       app_switch                      0.030                     0.644         0             0              1              0             0  

In [ ]:
# Temporal Gap Baseline

print("=" * 70)
print("SEGMENTATION MODEL — V0 TEMPORAL GAP BASELINE")
print("=" * 70)


# Work from the feature table created in Cell 4


v0_data = features[
    [
        "session_id",
        "timestamp",
        "event_type",
        "time_since_previous_event"
    ]
].copy()


# Candidate inactivity thresholds


THRESHOLDS = [
    1,
    2,
    3,
    5,
    10,
    15,
    20
]


# Create candidate boundaries


for threshold in THRESHOLDS:

    column_name = (
        f"boundary_gap_{threshold}s"
    )

    v0_data[column_name] = (
        v0_data["time_since_previous_event"]
        >= threshold
    ).astype(int)



# Summary


print("\nRAW EVENTS")
print("----------")
print(f"Total events: {len(v0_data):,}")
print(
    f"Sessions: "
    f"{v0_data['session_id'].nunique()}"
)


print("\nCANDIDATE THRESHOLD RESULTS")
print("---------------------------")

for threshold in THRESHOLDS:

    boundary_column = (
        f"boundary_gap_{threshold}s"
    )

    boundary_count = (
        v0_data[boundary_column].sum()
    )

    boundaries_per_session = (
        boundary_count /
        v0_data["session_id"].nunique()
    )

    print(
        f"{threshold:>2} sec → "
        f"{boundary_count:>6,} boundaries | "
        f"{boundaries_per_session:>7.2f} per session"
    )



# Select one temporary baseline


V0_THRESHOLD = 5

v0_data["v0_boundary"] = (
    v0_data["time_since_previous_event"]
    >= V0_THRESHOLD
).astype(int)


print("\nV0 BASELINE")
print("-----------")

print(
    f"Selected threshold: "
    f"{V0_THRESHOLD} seconds"
)

print(
    f"Predicted boundaries: "
    f"{v0_data['v0_boundary'].sum():,}"
)

print(
    f"Average boundaries/session: "
    f"{v0_data['v0_boundary'].sum() / v0_data['session_id'].nunique():.2f}"
)


print("\nV0 BASELINE CREATED")

SEGMENTATION MODEL — V0 TEMPORAL GAP BASELINE

RAW EVENTS
----------
Total events: 162,768
Sessions: 63

CANDIDATE THRESHOLD RESULTS
---------------------------
 1 sec → 18,536 boundaries |  294.22 per session
 2 sec → 10,623 boundaries |  168.62 per session
 3 sec →  5,145 boundaries |   81.67 per session
 5 sec →  2,059 boundaries |   32.68 per session
10 sec →  1,425 boundaries |   22.62 per session
15 sec →    888 boundaries |   14.10 per session
20 sec →    459 boundaries |    7.29 per session

V0 BASELINE
-----------
Selected threshold: 5 seconds
Predicted boundaries: 2,059
Average boundaries/session: 32.68

V0 BASELINE CREATED


In [24]:
# V0 boundary evaluation

print("V0 BOUNDARY EVALUATION")

gt_records = []

for gt_file in sorted(DATASET_A_DIR.rglob("gt.jsonl")):
    session_id = gt_file.parent.name

    with open(gt_file, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if not line:
                continue

            record = json.loads(line)

            if record.get("event") == "process_started":
                gt_records.append({
                    "session_id": session_id,
                    "timestamp": record.get("ts_utc"),
                    "process_code": record.get("process_code"),
                    "case_id": record.get("case_id")
                })

gt_boundaries = pd.DataFrame(gt_records)

gt_boundaries["timestamp"] = pd.to_datetime(
    gt_boundaries["timestamp"],
    utc=True,
    errors="coerce"
)

gt_boundaries = (
    gt_boundaries
    .drop_duplicates(
        subset=[
            "session_id",
            "timestamp",
            "process_code",
            "case_id"
        ]
    )
    .sort_values(
        ["session_id", "timestamp"]
    )
    .reset_index(drop=True)
)

v0_boundaries = (
    v0_data[
        v0_data["v0_boundary"] == 1
    ][
        ["session_id", "timestamp"]
    ]
    .sort_values(
        ["session_id", "timestamp"]
    )
    .reset_index(drop=True)
)

print(f"GT boundaries: {len(gt_boundaries):,}")
print(f"V0 predicted boundaries: {len(v0_boundaries):,}")


def evaluate_boundaries(predictions, ground_truth, tolerance_seconds):
    tp = 0
    fp = 0
    fn = 0

    sessions = ground_truth["session_id"].unique()

    for session_id in sessions:

        pred_times = predictions[
            predictions["session_id"] == session_id
        ]["timestamp"].tolist()

        gt_times = ground_truth[
            ground_truth["session_id"] == session_id
        ]["timestamp"].tolist()

        matched = set()

        for gt_time in gt_times:

            best_index = None
            best_distance = None

            for i, pred_time in enumerate(pred_times):

                if i in matched:
                    continue

                distance = abs(
                    (pred_time - gt_time).total_seconds()
                )

                if distance <= tolerance_seconds:

                    if (
                        best_distance is None
                        or distance < best_distance
                    ):
                        best_index = i
                        best_distance = distance

            if best_index is not None:
                tp += 1
                matched.add(best_index)

        fp += len(pred_times) - len(matched)
        fn += len(gt_times) - len(matched)

    precision = tp / (tp + fp) if tp + fp else 0
    recall = tp / (tp + fn) if tp + fn else 0
    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall
        else 0
    )

    return tp, fp, fn, precision, recall, f1


results = []

for tolerance in [0.5, 1, 2, 3, 5]:

    tp, fp, fn, precision, recall, f1 = evaluate_boundaries(
        v0_boundaries,
        gt_boundaries,
        tolerance
    )

    results.append({
        "tolerance_sec": tolerance,
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "precision": precision,
        "recall": recall,
        "F1": f1
    })

V0_RESULTS = pd.DataFrame(results)

print("\nV0 RESULTS")
print(V0_RESULTS.to_string(index=False))

best = V0_RESULTS.loc[
    V0_RESULTS["F1"].idxmax()
]

print("\nBEST V0 RESULT")
print(f"Tolerance: {best['tolerance_sec']} sec")
print(f"Precision: {best['precision']:.3f}")
print(f"Recall: {best['recall']:.3f}")
print(f"F1: {best['F1']:.3f}")

V0 BOUNDARY EVALUATION
GT boundaries: 1,819
V0 predicted boundaries: 2,059

V0 RESULTS
 tolerance_sec  TP   FP   FN  precision   recall       F1
           0.5 159 1900 1660   0.077222 0.087411 0.082001
           1.0 189 1870 1630   0.091792 0.103903 0.097473
           2.0 252 1807 1567   0.122390 0.138538 0.129964
           3.0 409 1650 1410   0.198640 0.224849 0.210933
           5.0 610 1449 1209   0.296260 0.335349 0.314595

BEST V0 RESULT
Tolerance: 5.0 sec
Precision: 0.296
Recall: 0.335
F1: 0.315


In [26]:
# Cell 7 — V1 boundary training labels

v1_data = features[
    [
        "session_id",
        "timestamp",
        "event_type",
        "layer",
        "time_since_previous_event",
        "time_since_session_start",
        "is_click",
        "is_keystroke",
        "is_app_switch",
        "is_screenshot",
        "is_clipboard",
        "is_browser"
    ]
].copy()

v1_data["boundary_label"] = 0

label_tolerance = 2.0

for session_id, group in gt_boundaries.groupby("session_id"):

    event_indices = v1_data.index[
        v1_data["session_id"] == session_id
    ]

    event_times = v1_data.loc[
        event_indices, "timestamp"
    ].sort_values()

    if len(event_times) == 0:
        continue

    for gt_time in group["timestamp"]:

        distances = (
            event_times - gt_time
        ).abs().dt.total_seconds()

        nearest_index = distances.idxmin()

        if distances.loc[nearest_index] <= label_tolerance:
            v1_data.loc[
                nearest_index,
                "boundary_label"
            ] = 1

print("V1 TRAINING LABELS")
print("-------------------")
print(f"Total events: {len(v1_data):,}")
print(f"Positive boundaries: {v1_data['boundary_label'].sum():,}")
print(
    f"Negative events: "
    f"{(v1_data['boundary_label'] == 0).sum():,}"
)
print(
    f"Positive rate: "
    f"{100 * v1_data['boundary_label'].mean():.3f}%"
)

print("\nLabel distribution:")
print(
    v1_data["boundary_label"]
    .value_counts()
    .sort_index()
    .to_string()
)

print("\nV1 LABEL DATASET READY")

V1 TRAINING LABELS
-------------------
Total events: 162,768
Positive boundaries: 1,545
Negative events: 161,223
Positive rate: 0.949%

Label distribution:
boundary_label
0    161223
1      1545

V1 LABEL DATASET READY


In [28]:
# Cell 8 — Session-level train/validation/test split

session_ids = (
    v1_data["session_id"]
    .dropna()
    .astype(str)
    .drop_duplicates()
    .tolist()
)

rng = random.Random(RANDOM_SEED)
rng.shuffle(session_ids)

n_sessions = len(session_ids)
n_train = int(n_sessions * 0.70)
n_validation = int(n_sessions * 0.15)

train_sessions = session_ids[:n_train]
validation_sessions = session_ids[
    n_train:n_train + n_validation
]
test_sessions = session_ids[
    n_train + n_validation:
]

train_data = v1_data[
    v1_data["session_id"].astype(str).isin(train_sessions)
].copy()

validation_data = v1_data[
    v1_data["session_id"].astype(str).isin(validation_sessions)
].copy()

test_data = v1_data[
    v1_data["session_id"].astype(str).isin(test_sessions)
].copy()

print("V1 DATA SPLIT")
print("-------------")

print(f"Total sessions: {n_sessions}")
print(f"Training sessions: {len(train_sessions)}")
print(f"Validation sessions: {len(validation_sessions)}")
print(f"Test sessions: {len(test_sessions)}")

print("\nEvent counts:")
print(f"Training: {len(train_data):,}")
print(f"Validation: {len(validation_data):,}")
print(f"Test: {len(test_data):,}")

print("\nPositive boundary counts:")
print(f"Training: {train_data['boundary_label'].sum():,}")
print(f"Validation: {validation_data['boundary_label'].sum():,}")
print(f"Test: {test_data['boundary_label'].sum():,}")

print("\nV1 SESSION-LEVEL SPLIT READY")

V1 DATA SPLIT
-------------
Total sessions: 63
Training sessions: 44
Validation sessions: 9
Test sessions: 10

Event counts:
Training: 115,978
Validation: 18,892
Test: 27,898

Positive boundary counts:
Training: 1,101
Validation: 170
Test: 274

V1 SESSION-LEVEL SPLIT READY


In [29]:
# Cell 9 — V1 feature preprocessing

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

feature_columns = [
    "event_type",
    "layer",
    "time_since_previous_event",
    "time_since_session_start",
    "is_click",
    "is_keystroke",
    "is_app_switch",
    "is_screenshot",
    "is_clipboard",
    "is_browser"
]

categorical_features = [
    "event_type",
    "layer"
]

numerical_features = [
    "time_since_previous_event",
    "time_since_session_start",
    "is_click",
    "is_keystroke",
    "is_app_switch",
    "is_screenshot",
    "is_clipboard",
    "is_browser"
]

X_train = train_data[feature_columns].copy()
y_train = train_data["boundary_label"].copy()

X_validation = validation_data[feature_columns].copy()
y_validation = validation_data["boundary_label"].copy()

X_test = test_data[feature_columns].copy()
y_test = test_data["boundary_label"].copy()

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        ),
        (
            "numerical",
            StandardScaler(),
            numerical_features
        )
    ]
)

X_train_processed = preprocessor.fit_transform(X_train)
X_validation_processed = preprocessor.transform(X_validation)
X_test_processed = preprocessor.transform(X_test)

print("V1 FEATURE PREPROCESSING")
print("------------------------")

print(f"Training samples: {X_train_processed.shape[0]:,}")
print(f"Validation samples: {X_validation_processed.shape[0]:,}")
print(f"Test samples: {X_test_processed.shape[0]:,}")

print(f"Processed features: {X_train_processed.shape[1]:,}")

print("\nPositive labels:")
print(f"Training: {y_train.sum():,}")
print(f"Validation: {y_validation.sum():,}")
print(f"Test: {y_test.sum():,}")

print("\nV1 PREPROCESSING READY")

V1 FEATURE PREPROCESSING
------------------------
Training samples: 115,978
Validation samples: 18,892
Test samples: 27,898
Processed features: 39

Positive labels:
Training: 1,101
Validation: 170
Test: 274

V1 PREPROCESSING READY


In [31]:
!pip install CatBoost

  Using cached catboost-1.2.10-cp314-cp314-win_amd64.whl.metadata (1.5 kB)
  Using cached graphviz-0.21-py3-none-any.whl.metadata (12 kB)
Using cached catboost-1.2.10-cp314-cp314-win_amd64.whl (101.7 MB)
Using cached graphviz-0.21-py3-none-any.whl (47 kB)

   ---------------------------------------- 0/2 [graphviz]
   ---------------------------------------- 0/2 [graphviz]
   ---------------------------------------- 0/2 [graphviz]
   ---------------------------------------- 0/2 [graphviz]
   -------------------- ------------------- 1/2 [CatBoost]
   -------------------- ------------------- 1/2 [CatBoost]
   -------------------- ------------------- 1/2 [CatBoost]
   -------------------- ------------------- 1/2 [CatBoost]
   -------------------- ------------------- 1/2 [CatBoost]
   -------------------- ------------------- 1/2 [CatBoost]
   -------------------- ------------------- 1/2 [CatBoost]
   -------------------- ------------------- 1/2 [CatBoost]
   -------------------- -----------

In [32]:
# Cell 10 — Train V1 classification models

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

positive_weight = (
    (y_train == 0).sum() / (y_train == 1).sum()
)

v1_models = {
    "Logistic Regression": LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        random_state=RANDOM_SEED
    ),

    "Decision Tree": DecisionTreeClassifier(
        class_weight="balanced",
        max_depth=12,
        min_samples_leaf=10,
        random_state=RANDOM_SEED
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        class_weight="balanced",
        max_depth=15,
        min_samples_leaf=5,
        n_jobs=-1,
        random_state=RANDOM_SEED
    ),

    "AdaBoost": AdaBoostClassifier(
        n_estimators=200,
        learning_rate=0.05,
        random_state=RANDOM_SEED
    ),

    "XGBoost": XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=positive_weight,
        eval_metric="logloss",
        random_state=RANDOM_SEED,
        n_jobs=-1
    ),

    "CatBoost": CatBoostClassifier(
        iterations=200,
        depth=6,
        learning_rate=0.05,
        loss_function="Logloss",
        auto_class_weights="Balanced",
        verbose=False,
        random_seed=RANDOM_SEED
    )
}

for name, model in v1_models.items():
    print(f"Training {name}...")
    model.fit(X_train_processed, y_train)

print("\nV1 MODEL TRAINING COMPLETE")
print("---------------------------")
print(f"Models trained: {len(v1_models)}")

for name in v1_models:
    print(f"✓ {name}")

Training Logistic Regression...
Training Decision Tree...
Training Random Forest...
Training AdaBoost...
Training XGBoost...
Training CatBoost...

V1 MODEL TRAINING COMPLETE
---------------------------
Models trained: 6
✓ Logistic Regression
✓ Decision Tree
✓ Random Forest
✓ AdaBoost
✓ XGBoost
✓ CatBoost


In [33]:
# Cell 11 — V1 model evaluation

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)

v1_results = []

for name, model in v1_models.items():

    train_prob = model.predict_proba(X_train_processed)[:, 1]
    validation_prob = model.predict_proba(X_validation_processed)[:, 1]

    train_pred = (train_prob >= 0.5).astype(int)
    validation_pred = (validation_prob >= 0.5).astype(int)

    train_f1 = f1_score(y_train, train_pred, zero_division=0)
    validation_f1 = f1_score(
        y_validation,
        validation_pred,
        zero_division=0
    )

    validation_precision = precision_score(
        y_validation,
        validation_pred,
        zero_division=0
    )

    validation_recall = recall_score(
        y_validation,
        validation_pred,
        zero_division=0
    )

    validation_roc_auc = roc_auc_score(
        y_validation,
        validation_prob
    )

    validation_pr_auc = average_precision_score(
        y_validation,
        validation_prob
    )

    v1_results.append({
        "Model": name,
        "Train F1": train_f1,
        "Validation F1": validation_f1,
        "Precision": validation_precision,
        "Recall": validation_recall,
        "ROC-AUC": validation_roc_auc,
        "PR-AUC": validation_pr_auc,
        "Predicted Boundaries": validation_pred.sum()
    })

v1_results_df = (
    pd.DataFrame(v1_results)
    .sort_values(
        "Validation F1",
        ascending=False
    )
    .reset_index(drop=True)
)

print("V1 MODEL COMPARISON")
print("-------------------")

print(
    v1_results_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

V1 MODEL COMPARISON
-------------------
              Model  Train F1  Validation F1  Precision  Recall  ROC-AUC  PR-AUC  Predicted Boundaries
      Random Forest    0.6113         0.6274     0.4635  0.9706   0.9942  0.8127                   356
            XGBoost    0.4739         0.4699     0.3106  0.9647   0.9942  0.8132                   528
      Decision Tree    0.4094         0.3858     0.2415  0.9588   0.9839  0.7765                   675
           CatBoost    0.3224         0.3451     0.2096  0.9765   0.9951  0.7971                   792
Logistic Regression    0.0510         0.0512     0.0263  0.9765   0.8768  0.1589                  6319
           AdaBoost    0.0000         0.0000     0.0000  0.0000   0.9423  0.1257                     0


In [34]:
# Cell 12 — Validation threshold optimization

thresholds = np.arange(0.10, 0.91, 0.05)

threshold_results = []

for name, model in v1_models.items():

    validation_prob = model.predict_proba(
        X_validation_processed
    )[:, 1]

    for threshold in thresholds:

        validation_pred = (
            validation_prob >= threshold
        ).astype(int)

        precision = precision_score(
            y_validation,
            validation_pred,
            zero_division=0
        )

        recall = recall_score(
            y_validation,
            validation_pred,
            zero_division=0
        )

        f1 = f1_score(
            y_validation,
            validation_pred,
            zero_division=0
        )

        threshold_results.append({
            "Model": name,
            "Threshold": round(float(threshold), 2),
            "Precision": precision,
            "Recall": recall,
            "F1": f1,
            "Predicted Boundaries": int(
                validation_pred.sum()
            )
        })

threshold_results_df = pd.DataFrame(
    threshold_results
)

best_thresholds = (
    threshold_results_df
    .sort_values(
        ["Model", "F1"],
        ascending=[True, False]
    )
    .groupby("Model")
    .head(1)
    .sort_values(
        "F1",
        ascending=False
    )
    .reset_index(drop=True)
)

print("BEST VALIDATION THRESHOLD BY MODEL")
print("===================================")

print(
    best_thresholds.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

BEST VALIDATION THRESHOLD BY MODEL
              Model  Threshold  Precision  Recall     F1  Predicted Boundaries
      Random Forest     0.9000     0.7718  0.9353 0.8457                   206
            XGBoost     0.9000     0.6763  0.9588 0.7932                   241
           CatBoost     0.9000     0.6667  0.9529 0.7845                   243
      Decision Tree     0.9000     0.5062  0.9588 0.6626                   322
           AdaBoost     0.3000     0.3305  0.2294 0.2708                   118
Logistic Regression     0.9000     0.2532  0.2294 0.2407                   154


In [35]:
# Cell 14 — V1 ANN / MLP classifier

from sklearn.neural_network import MLPClassifier

ann_model = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    activation="relu",
    solver="adam",
    alpha=0.0001,
    batch_size=256,
    learning_rate_init=0.001,
    max_iter=30,
    early_stopping=True,
    validation_fraction=0.15,
    n_iter_no_change=5,
    random_state=RANDOM_SEED
)

ann_model.fit(
    X_train_processed,
    y_train
)

print("ANN MODEL TRAINING")
print("-------------------")
print("Architecture: 39 → 64 → 32 → 1")
print("Activation: ReLU")
print("Optimizer: Adam")
print(f"Training samples: {X_train_processed.shape[0]:,}")
print(f"Input features: {X_train_processed.shape[1]:,}")
print(f"Iterations completed: {ann_model.n_iter_}")

print("\nANN MODEL TRAINED SUCCESSFULLY")

ANN MODEL TRAINING
-------------------
Architecture: 39 → 64 → 32 → 1
Activation: ReLU
Optimizer: Adam
Training samples: 115,978
Input features: 39
Iterations completed: 10

ANN MODEL TRAINED SUCCESSFULLY


In [36]:
# Cell 15 — ANN evaluation

ann_train_prob = ann_model.predict_proba(
    X_train_processed
)[:, 1]

ann_validation_prob = ann_model.predict_proba(
    X_validation_processed
)[:, 1]

ann_train_pred = (
    ann_train_prob >= 0.5
).astype(int)

ann_validation_pred = (
    ann_validation_prob >= 0.5
).astype(int)

ann_train_f1 = f1_score(
    y_train,
    ann_train_pred,
    zero_division=0
)

ann_validation_f1 = f1_score(
    y_validation,
    ann_validation_pred,
    zero_division=0
)

ann_precision = precision_score(
    y_validation,
    ann_validation_pred,
    zero_division=0
)

ann_recall = recall_score(
    y_validation,
    ann_validation_pred,
    zero_division=0
)

ann_roc_auc = roc_auc_score(
    y_validation,
    ann_validation_prob
)

ann_pr_auc = average_precision_score(
    y_validation,
    ann_validation_prob
)

print("ANN MODEL EVALUATION")
print("====================")

print(f"Train F1:       {ann_train_f1:.4f}")
print(f"Validation F1:  {ann_validation_f1:.4f}")
print(f"Precision:      {ann_precision:.4f}")
print(f"Recall:         {ann_recall:.4f}")
print(f"ROC-AUC:        {ann_roc_auc:.4f}")
print(f"PR-AUC:         {ann_pr_auc:.4f}")
print(
    f"Predicted boundaries: "
    f"{ann_validation_pred.sum()}"
)

ANN MODEL EVALUATION
Train F1:       0.3757
Validation F1:  0.3562
Precision:      0.7959
Recall:         0.2294
ROC-AUC:        0.9914
PR-AUC:         0.7413
Predicted boundaries: 49


In [37]:
# Cell 16 — ANN threshold optimization

ann_thresholds = np.arange(0.10, 1.00, 0.01)

ann_threshold_results = []

for threshold in ann_thresholds:

    validation_pred = (
        ann_validation_prob >= threshold
    ).astype(int)

    precision = precision_score(
        y_validation,
        validation_pred,
        zero_division=0
    )

    recall = recall_score(
        y_validation,
        validation_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_validation,
        validation_pred,
        zero_division=0
    )

    ann_threshold_results.append({
        "Threshold": round(float(threshold), 2),
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "Predicted Boundaries": int(
            validation_pred.sum()
        )
    })

ann_threshold_results_df = pd.DataFrame(
    ann_threshold_results
)

best_ann = (
    ann_threshold_results_df
    .sort_values("F1", ascending=False)
    .iloc[0]
)

print("ANN BEST THRESHOLD")
print("==================")

print(
    best_ann.to_frame().T.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

ANN BEST THRESHOLD
 Threshold  Precision  Recall     F1  Predicted Boundaries
    0.2100     0.5887  0.9176 0.7172              265.0000


In [38]:
# Cell 17 — Random Forest boundary predictions

BEST_MODEL_NAME = "Random Forest"
BEST_THRESHOLD = 0.90

best_model = v1_models[BEST_MODEL_NAME]

test_probabilities = best_model.predict_proba(
    X_test_processed
)[:, 1]

test_predictions = (
    test_probabilities >= BEST_THRESHOLD
).astype(int)

boundary_predictions = test_data[
    [
        "session_id",
        "timestamp"
    ]
].copy()

boundary_predictions["boundary_probability"] = test_probabilities
boundary_predictions["boundary_prediction"] = test_predictions

boundary_predictions = (
    boundary_predictions
    .sort_values(
        ["session_id", "timestamp"]
    )
    .reset_index(drop=True)
)

print("RANDOM FOREST — BOUNDARY PREDICTIONS")
print("=====================================")

print(f"Model: {BEST_MODEL_NAME}")
print(f"Threshold: {BEST_THRESHOLD}")

print(f"\nTest events: {len(boundary_predictions):,}")

print(
    f"Predicted boundaries: "
    f"{boundary_predictions['boundary_prediction'].sum():,}"
)

print(
    f"Boundary rate: "
    f"{100 * boundary_predictions['boundary_prediction'].mean():.3f}%"
)

print("\nSample boundary predictions:")

print(
    boundary_predictions[
        boundary_predictions["boundary_prediction"] == 1
    ].head(10).to_string(index=False)
)

print("\nBOUNDARY PREDICTIONS READY")

RANDOM FOREST — BOUNDARY PREDICTIONS
Model: Random Forest
Threshold: 0.9

Test events: 27,898
Predicted boundaries: 316
Boundary rate: 1.133%

Sample boundary predictions:
                       session_id                        timestamp  boundary_probability  boundary_prediction
ses_20260630-124826-CHAITANYA0BCF 2026-06-30 12:49:20.714000+00:00              0.991329                    1
ses_20260630-124826-CHAITANYA0BCF 2026-06-30 12:49:29.658000+00:00              0.990654                    1
ses_20260630-124826-CHAITANYA0BCF 2026-06-30 12:50:00.360000+00:00              0.997172                    1
ses_20260630-124826-CHAITANYA0BCF 2026-06-30 12:51:00.935000+00:00              0.979968                    1
ses_20260630-124826-CHAITANYA0BCF 2026-06-30 12:51:26.410000+00:00              0.989195                    1
ses_20260630-124826-CHAITANYA0BCF 2026-06-30 12:52:05.573000+00:00              0.905405                    1
ses_20260630-124826-CHAITANYA0BCF 2026-06-30 12:52:08.6860

In [40]:
# Cell 18 — Temporal post-processing of predicted boundaries

MIN_BOUNDARY_GAP_SECONDS = 5

predicted_boundaries = boundary_predictions[
    boundary_predictions["boundary_prediction"] == 1
].copy()

predicted_boundaries = predicted_boundaries.sort_values(
    ["session_id", "timestamp"]
)

filtered_boundary_records = []

for session_id, group in predicted_boundaries.groupby("session_id"):

    group = group.sort_values("timestamp")

    last_selected_time = None

    for _, row in group.iterrows():

        current_time = row["timestamp"]

        if last_selected_time is None:
            filtered_boundary_records.append({
                "session_id": session_id,
                "timestamp": current_time,
                "boundary_probability": row["boundary_probability"]
            })

            last_selected_time = current_time

        else:
            gap_seconds = (
                current_time - last_selected_time
            ).total_seconds()

            if gap_seconds >= MIN_BOUNDARY_GAP_SECONDS:
                filtered_boundary_records.append({
                    "session_id": session_id,
                    "timestamp": current_time,
                    "boundary_probability": row["boundary_probability"]
                })

                last_selected_time = current_time

filtered_boundaries = pd.DataFrame(
    filtered_boundary_records
)

original_count = len(predicted_boundaries)
filtered_count = len(filtered_boundaries)

print("TEMPORAL POST-PROCESSING")
print("========================")

print(f"Original predicted boundaries: {original_count:,}")
print(f"After minimum-gap filtering: {filtered_count:,}")
print(f"Removed boundaries: {original_count - filtered_count:,}")
print(f"Minimum boundary gap: {MIN_BOUNDARY_GAP_SECONDS} seconds")

print("\nFILTERED BOUNDARY SAMPLE")

print(
    filtered_boundaries
    .head(15)
    .to_string(index=False)
)

print("\nTEMPORAL POST-PROCESSING COMPLETE")

TEMPORAL POST-PROCESSING
Original predicted boundaries: 316
After minimum-gap filtering: 293
Removed boundaries: 23
Minimum boundary gap: 5 seconds

FILTERED BOUNDARY SAMPLE
                       session_id                        timestamp  boundary_probability
ses_20260630-124826-CHAITANYA0BCF 2026-06-30 12:49:20.714000+00:00              0.991329
ses_20260630-124826-CHAITANYA0BCF 2026-06-30 12:49:29.658000+00:00              0.990654
ses_20260630-124826-CHAITANYA0BCF 2026-06-30 12:50:00.360000+00:00              0.997172
ses_20260630-124826-CHAITANYA0BCF 2026-06-30 12:51:00.935000+00:00              0.979968
ses_20260630-124826-CHAITANYA0BCF 2026-06-30 12:51:26.410000+00:00              0.989195
ses_20260630-124826-CHAITANYA0BCF 2026-06-30 12:52:05.573000+00:00              0.905405
ses_20260630-124826-CHAITANYA0BCF 2026-06-30 12:53:12.494000+00:00              0.987490
ses_20260630-124826-CHAITANYA0BCF 2026-06-30 12:53:36.533000+00:00              0.988770
ses_20260630-124826-CHAIT

In [41]:
# Cell 19 — Construct predicted segments

segment_records = []

for session_id, group in filtered_boundaries.groupby("session_id"):

    boundaries = (
        group
        .sort_values("timestamp")
        .reset_index(drop=True)
    )

    session_events = (
        test_data[
            test_data["session_id"] == session_id
        ]
        .sort_values("timestamp")
    )

    if session_events.empty:
        continue

    session_start = session_events["timestamp"].min()
    session_end = session_events["timestamp"].max()

    boundary_times = boundaries["timestamp"].tolist()

    if not boundary_times:
        continue

    if boundary_times[0] > session_start:
        segment_records.append({
            "session_id": session_id,
            "start": session_start,
            "end": boundary_times[0]
        })

    for i in range(len(boundary_times) - 1):
        start_time = boundary_times[i]
        end_time = boundary_times[i + 1]

        if end_time > start_time:
            segment_records.append({
                "session_id": session_id,
                "start": start_time,
                "end": end_time
            })

    if boundary_times[-1] < session_end:
        segment_records.append({
            "session_id": session_id,
            "start": boundary_times[-1],
            "end": session_end
        })

predicted_segments = pd.DataFrame(segment_records)

print("PREDICTED SEGMENT CONSTRUCTION")
print("==============================")
print(f"Sessions: {predicted_segments['session_id'].nunique()}")
print(f"Predicted segments: {len(predicted_segments):,}")

print("\nSegments per session:")
print(
    predicted_segments
    .groupby("session_id")
    .size()
    .describe()
    .to_string()
)

print("\nSample predicted segments:")
print(
    predicted_segments
    .head(10)
    .to_string(index=False)
)

print("\nPREDICTED SEGMENTS READY")

PREDICTED SEGMENT CONSTRUCTION
Sessions: 10
Predicted segments: 303

Segments per session:
count    10.000000
mean     30.300000
std       3.743142
min      25.000000
25%      27.500000
50%      30.000000
75%      32.250000
max      37.000000

Sample predicted segments:
                       session_id                            start                              end
ses_20260630-124826-CHAITANYA0BCF 2026-06-30 12:48:26.691000+00:00 2026-06-30 12:49:20.714000+00:00
ses_20260630-124826-CHAITANYA0BCF 2026-06-30 12:49:20.714000+00:00 2026-06-30 12:49:29.658000+00:00
ses_20260630-124826-CHAITANYA0BCF 2026-06-30 12:49:29.658000+00:00 2026-06-30 12:50:00.360000+00:00
ses_20260630-124826-CHAITANYA0BCF 2026-06-30 12:50:00.360000+00:00 2026-06-30 12:51:00.935000+00:00
ses_20260630-124826-CHAITANYA0BCF 2026-06-30 12:51:00.935000+00:00 2026-06-30 12:51:26.410000+00:00
ses_20260630-124826-CHAITANYA0BCF 2026-06-30 12:51:26.410000+00:00 2026-06-30 12:52:05.573000+00:00
ses_20260630-124826-CHAITANYA

In [42]:
# Cell 20 — Evaluate predicted boundaries against ground truth

print("FINAL V1 TEST — BOUNDARY EVALUATION")
print("====================================")

# Use only GT boundaries belonging to the unseen test sessions
test_gt_boundaries = gt_boundaries[
    gt_boundaries["session_id"].isin(test_sessions)
].copy()

test_predicted_boundaries = filtered_boundaries[
    filtered_boundaries["session_id"].isin(test_sessions)
].copy()

print(f"Test sessions: {len(test_sessions)}")
print(f"GT boundaries: {len(test_gt_boundaries):,}")
print(f"Predicted boundaries: {len(test_predicted_boundaries):,}")


def evaluate_test_boundaries(predictions, ground_truth, tolerance_seconds):

    tp = 0
    fp = 0
    fn = 0

    for session_id in ground_truth["session_id"].unique():

        pred_times = (
            predictions[
                predictions["session_id"] == session_id
            ]["timestamp"]
            .sort_values()
            .tolist()
        )

        gt_times = (
            ground_truth[
                ground_truth["session_id"] == session_id
            ]["timestamp"]
            .sort_values()
            .tolist()
        )

        matched_predictions = set()

        for gt_time in gt_times:

            best_index = None
            best_distance = None

            for i, pred_time in enumerate(pred_times):

                if i in matched_predictions:
                    continue

                distance = abs(
                    (pred_time - gt_time).total_seconds()
                )

                if distance <= tolerance_seconds:

                    if (
                        best_distance is None
                        or distance < best_distance
                    ):
                        best_index = i
                        best_distance = distance

            if best_index is not None:
                tp += 1
                matched_predictions.add(best_index)

        fp += len(pred_times) - len(matched_predictions)
        fn += len(gt_times) - len(matched_predictions)

    precision = tp / (tp + fp) if (tp + fp) else 0
    recall = tp / (tp + fn) if (tp + fn) else 0

    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall)
        else 0
    )

    return tp, fp, fn, precision, recall, f1


test_results = []

for tolerance in [0.5, 1, 2, 3, 5]:

    tp, fp, fn, precision, recall, f1 = evaluate_test_boundaries(
        test_predicted_boundaries,
        test_gt_boundaries,
        tolerance
    )

    test_results.append({
        "Tolerance (sec)": tolerance,
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "Precision": precision,
        "Recall": recall,
        "F1": f1
    })


V1_TEST_RESULTS = pd.DataFrame(test_results)

print("\nTEST BOUNDARY RESULTS")
print("---------------------")

print(
    V1_TEST_RESULTS.to_string(
        index=False,
        formatters={
            "Precision": "{:.3f}".format,
            "Recall": "{:.3f}".format,
            "F1": "{:.3f}".format
        }
    )
)

print("\nV1 TEST EVALUATION COMPLETE")

FINAL V1 TEST — BOUNDARY EVALUATION
Test sessions: 10
GT boundaries: 303
Predicted boundaries: 293

TEST BOUNDARY RESULTS
---------------------
 Tolerance (sec)  TP  FP  FN Precision Recall    F1
             0.5 142 151 161     0.485  0.469 0.477
             1.0 176 117 127     0.601  0.581 0.591
             2.0 217  76  86     0.741  0.716 0.728
             3.0 222  71  81     0.758  0.733 0.745
             5.0 223  70  80     0.761  0.736 0.748

V1 TEST EVALUATION COMPLETE


In [43]:
# Cell 21 — Inspect available context fields for V2

context_keywords = [
    "app",
    "window",
    "browser",
    "title",
    "process"
]

context_columns = [
    column
    for column in ALL_EVENTS_A.columns
    if any(keyword in column.lower() for keyword in context_keywords)
]

print("V2 CONTEXT FIELD INSPECTION")
print("===========================")

print(f"Total raw event columns: {len(ALL_EVENTS_A.columns)}")
print(f"Potential context columns: {len(context_columns)}")

print("\nContext-related columns:")
for column in context_columns:
    print(column)

print("\nV2 CONTEXT INSPECTION COMPLETE")

V2 CONTEXT FIELD INSPECTION
Total raw event columns: 410
Potential context columns: 50

Context-related columns:
context.active_app
context.active_browser_tab
context.visible_windows
context.open_apps
payload.capture_settings_snapshot.app_switching
payload.capture_settings_snapshot.blocked_apps
payload.capture_settings_snapshot.blocked_window_patterns
payload.capture_settings_snapshot.browser_alert
payload.capture_settings_snapshot.browser_click
payload.capture_settings_snapshot.browser_error
payload.capture_settings_snapshot.browser_form_input
payload.capture_settings_snapshot.browser_navigation
payload.capture_settings_snapshot.browser_tab_event
payload.capture_settings_snapshot.window_state_changes
payload.capture_settings_snapshot.window_title_changes
context.active_app.process_name
context.active_app.app_name
context.active_app.window_title
context.active_app.process_id
payload.new_app.app_name
payload.new_app.process_id
payload.new_app.process_name
payload.new_app.window_title
pa

In [44]:
# Cell 21 — Final V1 segmentation sanity check

print("FINAL V1 SEGMENTATION SANITY CHECK")
print("===================================")

# Calculate segment durations
final_check = predicted_segments.copy()

final_check["duration_seconds"] = (
    final_check["end"] - final_check["start"]
).dt.total_seconds()

print(f"Test sessions: {final_check['session_id'].nunique()}")
print(f"Predicted segments: {len(final_check):,}")

print("\nSEGMENT DURATION STATISTICS")
print("---------------------------")
print(
    final_check["duration_seconds"]
    .describe()
    .to_string()
)

print("\nVERY SHORT SEGMENTS")
print("-------------------")
short_segments = final_check[
    final_check["duration_seconds"] < 5
]

print(f"Segments < 5 sec: {len(short_segments):,}")
print(
    f"Percentage: "
    f"{100 * len(short_segments) / len(final_check):.2f}%"
)

print("\nVERY LONG SEGMENTS")
print("------------------")
long_segments = final_check[
    final_check["duration_seconds"] > 300
]

print(f"Segments > 5 min: {len(long_segments):,}")
print(
    f"Percentage: "
    f"{100 * len(long_segments) / len(final_check):.2f}%"
)

print("\nSEGMENTS PER TEST SESSION")
print("-------------------------")

session_summary = (
    final_check
    .groupby("session_id")
    .agg(
        segments=("session_id", "size"),
        total_time_seconds=("duration_seconds", "sum"),
        median_duration_seconds=("duration_seconds", "median")
    )
)

print(session_summary.to_string())

print("\nFINAL V1 DECISION")
print("-----------------")
print("Model: Random Forest")
print("Boundary threshold: 0.90")
print("Minimum boundary gap: 5 seconds")
print("Test boundary F1 @ 3 sec tolerance: 0.745")

print("\nSANITY CHECK COMPLETE")


FINAL V1 SEGMENTATION SANITY CHECK
Test sessions: 10
Predicted segments: 303

SEGMENT DURATION STATISTICS
---------------------------
count    303.000000
mean      47.719314
std       38.589313
min        0.964000
25%       25.371000
50%       34.542000
75%       60.942500
max      293.803000

VERY SHORT SEGMENTS
-------------------
Segments < 5 sec: 2
Percentage: 0.66%

VERY LONG SEGMENTS
------------------
Segments > 5 min: 0
Percentage: 0.00%

SEGMENTS PER TEST SESSION
-------------------------
                                     segments  total_time_seconds  median_duration_seconds
session_id                                                                                
ses_20260630-124826-CHAITANYA0BCF          30            1376.601                  30.4730
ses_20260630-135451-CHAITANYA0BCF          30            1267.733                  40.3755
ses_20260630-145433-JAYESH                 37            1443.567                  29.4290
ses_20260630-163424-SIDDHIGUPTAB00B       

In [45]:
# Cell 22 — Train and save final segmentation model

import joblib

print("FINAL SEGMENTATION MODEL")
print("========================")

# Combine training and validation data
final_train_data = pd.concat(
    [train_data, validation_data],
    axis=0
).sort_index()

X_final_train = final_train_data[feature_columns].copy()
y_final_train = final_train_data["boundary_label"].copy()

print(f"Training sessions: {len(train_sessions)}")
print(f"Validation sessions: {len(validation_sessions)}")
print(f"Final training sessions: {len(train_sessions) + len(validation_sessions)}")
print(f"Final training events: {len(final_train_data):,}")
print(f"Positive boundary labels: {y_final_train.sum():,}")

# Refit preprocessing on Train + Validation
final_preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        ),
        (
            "numerical",
            StandardScaler(),
            numerical_features
        )
    ]
)

X_final_train_processed = final_preprocessor.fit_transform(
    X_final_train
)

# Train final Random Forest
final_model = RandomForestClassifier(
    n_estimators=200,
    class_weight="balanced",
    max_depth=15,
    min_samples_leaf=5,
    n_jobs=-1,
    random_state=RANDOM_SEED
)

final_model.fit(
    X_final_train_processed,
    y_final_train
)

# Save model
model_path = MODELS_DIR / "best_segmentation_model.pkl"

joblib.dump(
    final_model,
    model_path
)

# Save preprocessing separately
preprocessor_path = PREPROCESSING_DIR / "best_preprocessor.pkl"

joblib.dump(
    final_preprocessor,
    preprocessor_path
)

print("\nFINAL MODEL")
print("-----------")
print("Model: Random Forest")
print("Boundary threshold: 0.90")
print("Minimum boundary gap: 5 seconds")
print(f"Model saved: {model_path.resolve()}")
print(f"Preprocessor saved: {preprocessor_path.resolve()}")

print("\nFINAL SEGMENTATION MODEL SAVED")

FINAL SEGMENTATION MODEL
Training sessions: 44
Validation sessions: 9
Final training sessions: 53
Final training events: 134,870
Positive boundary labels: 1,271

FINAL MODEL
-----------
Model: Random Forest
Boundary threshold: 0.90
Minimum boundary gap: 5 seconds
Model saved: C:\Users\Shambhavi Singh\OneDrive\Desktop\IBY_PROJECT\artifacts\models\best_segmentation_model.pkl
Preprocessor saved: C:\Users\Shambhavi Singh\OneDrive\Desktop\IBY_PROJECT\artifacts\preprocessing\best_preprocessor.pkl

FINAL SEGMENTATION MODEL SAVED


In [46]:
# Cell 23 — Verify saved model artifacts

import joblib

print("MODEL ARTIFACT VERIFICATION")
print("===========================")

# Load saved artifacts from disk
loaded_model = joblib.load(
    MODELS_DIR / "best_segmentation_model.pkl"
)

loaded_preprocessor = joblib.load(
    PREPROCESSING_DIR / "best_preprocessor.pkl"
)

print("Model loaded successfully:")
print(type(loaded_model).__name__)

print("\nPreprocessor loaded successfully:")
print(type(loaded_preprocessor).__name__)

print("\nModel parameters:")
print(f"Number of trees: {loaded_model.n_estimators}")
print(f"Maximum depth: {loaded_model.max_depth}")
print(f"Minimum samples per leaf: {loaded_model.min_samples_leaf}")

print("\nPreprocessor:")
print(f"Input features: {len(feature_columns)}")
print(
    f"Processed feature count: "
    f"{loaded_preprocessor.transform(X_final_train.head(1)).shape[1]}"
)

print("\nSegmentation configuration:")
print("Boundary threshold: 0.90")
print("Minimum boundary gap: 5 seconds")

print("\nMODEL ARTIFACT VERIFICATION COMPLETE")

MODEL ARTIFACT VERIFICATION
Model loaded successfully:
RandomForestClassifier

Preprocessor loaded successfully:
ColumnTransformer

Model parameters:
Number of trees: 200
Maximum depth: 15
Minimum samples per leaf: 5

Preprocessor:
Input features: 10
Processed feature count: 39

Segmentation configuration:
Boundary threshold: 0.90
Minimum boundary gap: 5 seconds

MODEL ARTIFACT VERIFICATION COMPLETE


In [47]:
# Cell 24 — Save final model metadata

import json

model_metadata = {
    "model_name": "RandomForestClassifier",
    "model_version": "V1",
    "random_seed": RANDOM_SEED,
    "boundary_threshold": 0.90,
    "minimum_boundary_gap_seconds": 5,
    "n_estimators": 200,
    "max_depth": 15,
    "min_samples_leaf": 5,
    "input_features": feature_columns,
    "categorical_features": categorical_features,
    "numerical_features": numerical_features,
    "training_sessions": len(train_sessions) + len(validation_sessions),
    "held_out_test_sessions": len(test_sessions),
    "test_boundary_f1_at_3_seconds": 0.745,
    "test_boundary_precision_at_3_seconds": 0.758,
    "test_boundary_recall_at_3_seconds": 0.733
}

metadata_path = METADATA_DIR / "segmentation_model_metadata.json"

with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(
        model_metadata,
        f,
        indent=4
    )

print("FINAL MODEL METADATA")
print("====================")
print(f"Metadata saved: {metadata_path.resolve()}")

print("\nConfiguration:")
print(f"Model: {model_metadata['model_name']}")
print(f"Version: {model_metadata['model_version']}")
print(f"Boundary threshold: {model_metadata['boundary_threshold']}")
print(
    f"Minimum boundary gap: "
    f"{model_metadata['minimum_boundary_gap_seconds']} seconds"
)
print(f"Input features: {len(model_metadata['input_features'])}")

print("\nFINAL MODEL PACKAGE READY")

FINAL MODEL METADATA
Metadata saved: C:\Users\Shambhavi Singh\OneDrive\Desktop\IBY_PROJECT\artifacts\metadata\segmentation_model_metadata.json

Configuration:
Model: RandomForestClassifier
Version: V1
Boundary threshold: 0.9
Minimum boundary gap: 5 seconds
Input features: 10

FINAL MODEL PACKAGE READY
